In [0]:
%sql
CREATE OR REPLACE TABLE silver_entity
AS
SELECT
  run_id,
  pdb_code,
  source_url,
  ingested_at,
  entity_id,
  details,
  formula_weight,
  pdbx_description,
  pdbx_ec,
  pdbx_fragment,
  pdbx_mutation,
  pdbx_number_of_molecules,
  src_method,
  entity_type
FROM (
  SELECT
    run_id,
    lower(pdb_code) AS pdb_code,
    source_url,
    ingested_at,
    entity_id,
    details,
    CAST(formula_weight AS DOUBLE) AS formula_weight,
    pdbx_description,
    pdbx_ec,
    pdbx_fragment,
    pdbx_mutation,
    CAST(pdbx_number_of_molecules AS INT) AS pdbx_number_of_molecules,
    src_method,
    entity_type,
    ROW_NUMBER() OVER (
      PARTITION BY lower(pdb_code), entity_id
      ORDER BY ingested_at DESC
    ) AS rn
  FROM pdb_lab.bronze_entity
)
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE pdb_lab.silver_exptl
USING DELTA AS
SELECT
  run_id,
  pdb_code,
  source_url,
  ingested_at,
  entry_id,
  method
FROM (
  SELECT
    run_id,
    lower(pdb_code) AS pdb_code,
    source_url,
    ingested_at,
    entry_id,
    method,
    ROW_NUMBER() OVER (
      PARTITION BY lower(pdb_code), entry_id
      ORDER BY ingested_at DESC
    ) AS rn
  FROM pdb_lab.bronze_exptl
)
WHERE rn = 1;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS pdb_lab.data_quality_warnings (
  warning_ts TIMESTAMP,
  table_name STRING,
  rule_name STRING,
  pdb_code STRING,
  entry_id STRING,
  invalid_value STRING,
  message STRING
) USING DELTA;

In [0]:
%sql
INSERT INTO pdb_lab.data_quality_warnings
SELECT
  current_timestamp() AS warning_ts,
  'pdb_lab.silver_exptl' AS table_name,
  'valid_exptl_method' AS rule_name,
  pdb_code,
  entry_id,
  method AS invalid_value,
  'Method is outside allowed controlled vocabulary' AS message
FROM pdb_lab.silver_exptl
WHERE upper(method) NOT IN (
  'ELECTRON CRYSTALLOGRAPHY',
  'ELECTRON MICROSCOPY',
  'EPR',
  'FIBER DIFFRACTION',
  'FLUORESCENCE TRANSFER',
  'INFRARED SPECTROSCOPY',
  'NEUTRON DIFFRACTION',
  'POWDER DIFFRACTION',
  'SOLID-STATE NMR',
  'SOLUTION NMR',
  'SOLUTION SCATTERING',
  'THEORETICAL MODEL',
  'X-RAY DIFFRACTION'
);

In [0]:
%sql
CREATE OR REPLACE TABLE silver_entity_poly_seq
AS
SELECT
  run_id,
  pdb_code,
  source_url,
  ingested_at,
  entity_id,
  mon_id,
  seq_num,
  hetero
FROM (
  SELECT
    run_id,
    lower(pdb_code) AS pdb_code,
    source_url,
    ingested_at,
    entity_id,
    mon_id,
    CAST(seq_num AS INT) AS seq_num,
    hetero,
    ROW_NUMBER() OVER (
      PARTITION BY lower(pdb_code), entity_id, mon_id, CAST(seq_num AS INT)
      ORDER BY ingested_at DESC
    ) AS rn
  FROM pdb_lab.bronze_entity_poly_seq
)
WHERE rn = 1;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_chem_comp
AS
SELECT
  run_id,
  pdb_code,
  source_url,
  ingested_at,
  chem_comp_id,
  formula,
  formula_weight,
  model_details,
  model_erf,
  model_source,
  mon_nstd_class,
  mon_nstd_details,
  mon_nstd_flag,
  mon_nstd_parent,
  mon_nstd_parent_comp_id,
  name,
  number_atoms_all,
  number_atoms_nh,
  one_letter_code,
  pdbx_ambiguous_flag,
  pdbx_casnum,
  pdbx_class_1,
  pdbx_class_2,
  pdbx_comp_type,
  pdbx_component_no,
  pdbx_formal_charge,
  pdbx_ideal_coordinates_details,
  pdbx_ideal_coordinates_missing_flag,
  pdbx_initial_date,
  pdbx_model_coordinates_db_code,
  pdbx_model_coordinates_details,
  pdbx_model_coordinates_missing_flag,
  pdbx_modification_details,
  pdbx_modified_date,
  pdbx_nscnum,
  pdbx_number_subcomponents,
  pdbx_pcm,
  pdbx_processing_site,
  pdbx_release_status,
  pdbx_replaced_by,
  pdbx_replaces,
  pdbx_reserved_name,
  pdbx_smiles,
  pdbx_status,
  pdbx_subcomponent_list,
  pdbx_synonyms,
  pdbx_type,
  pdbx_type_modified,
  three_letter_code,
  chem_comp_type
FROM (
  SELECT
    run_id,
    lower(pdb_code) AS pdb_code,
    source_url,
    ingested_at,
    chem_comp_id,
    formula,
    CAST(formula_weight AS DOUBLE) AS formula_weight,
    model_details,
    model_erf,
    model_source,
    mon_nstd_class,
    mon_nstd_details,
    mon_nstd_flag,
    mon_nstd_parent,
    mon_nstd_parent_comp_id,
    name,
    CAST(number_atoms_all AS INT) AS number_atoms_all,
    CAST(number_atoms_nh AS INT) AS number_atoms_nh,
    one_letter_code,
    pdbx_ambiguous_flag,
    pdbx_casnum,
    pdbx_class_1,
    pdbx_class_2,
    pdbx_comp_type,
    pdbx_component_no,
    CAST(pdbx_formal_charge AS INT) AS pdbx_formal_charge,
    pdbx_ideal_coordinates_details,
    pdbx_ideal_coordinates_missing_flag,
    TO_DATE(pdbx_initial_date) AS pdbx_initial_date,
    pdbx_model_coordinates_db_code,
    pdbx_model_coordinates_details,
    pdbx_model_coordinates_missing_flag,
    pdbx_modification_details,
    TO_DATE(pdbx_modified_date) AS pdbx_modified_date,
    pdbx_nscnum,
    CAST(pdbx_number_subcomponents AS INT) AS pdbx_number_subcomponents,
    pdbx_pcm,
    pdbx_processing_site,
    pdbx_release_status,
    pdbx_replaced_by,
    pdbx_replaces,
    pdbx_reserved_name,
    pdbx_smiles,
    pdbx_status,
    pdbx_subcomponent_list,
    pdbx_synonyms,
    pdbx_type,
    pdbx_type_modified,
    three_letter_code,
    chem_comp_type,
    ROW_NUMBER() OVER (
      PARTITION BY lower(pdb_code), chem_comp_id
      ORDER BY ingested_at DESC
    ) AS rn
  FROM pdb_lab.bronze_chem_comp
)
WHERE rn = 1;

In [0]:
%sql
CREATE OR REFRESH LIVE TABLE silver_pdbx_database_pdb_obs_spr
AS
SELECT
  run_id,
  pdb_code,
  source_url,
  ingested_at,
  obs_pdb_id,
  replace_pdb_id,
  obs_date,
  details,
  obs_id
FROM (
  SELECT
    run_id,
    lower(pdb_code) AS pdb_code,
    source_url,
    ingested_at,
    obs_pdb_id,
    replace_pdb_id,
    TO_DATE(obs_date) AS obs_date,
    details,
    obs_id,
    ROW_NUMBER() OVER (
      PARTITION BY lower(pdb_code), obs_id
      ORDER BY ingested_at DESC
    ) AS rn
  FROM pdb_lab.bronze_pdbx_database_pdb_obs_spr
)
WHERE rn = 1;